# 02. Data Cleaning: Telco Customer Churn

**Before this notebook:** read `notes/02_data_cleaning.md`.

**Goal:** fix the problems we found in notebook 01 and check for others, then save a clean file for the next notebooks.

**Steps**
1. Load the raw data
2. A tiny practice example (so the tools make sense)
3. Find missing values, including hidden ones
4. Fix `TotalCharges`
5. Check for duplicates
6. Check for inconsistent category values
7. Verify and save the cleaned data
8. Exercise

## Step 1: Load the raw data

Same as before. We always start from the **raw** file and never overwrite it.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("../../data/raw/telco_customer_churn.csv")
CLEAN_PATH = Path("../../data/processed/telco_clean.csv")

df = pd.read_csv(RAW_PATH)
print("Shape:", df.shape)

Shape: (7043, 21)


## Step 2: Practice on a tiny example

Before using the tools on 7,043 rows, let's see what they do on a table small enough to read.

In [2]:
toy = pd.DataFrame({
    "name": ["Ana", "Ben", "Ben", "Cara"],
    "age":  [25, np.nan, np.nan, 31],
    "city": ["Paris", "Rome", "Rome", " "],     # Cara's city is a hidden blank
})

print("isna():")
print(toy.isna())                 # True where Pandas sees a missing value
print()
print("Missing per column:")
print(toy.isna().sum())
print()
print("Duplicate rows (True = repeats an earlier row):")
print(toy.duplicated())
print()
print("Age after filling with the median:")
print(toy["age"].fillna(toy["age"].median()))

isna():
    name    age   city
0  False  False  False
1  False   True  False
2  False   True  False
3  False  False  False

Missing per column:
name    0
age     2
city    0
dtype: int64

Duplicate rows (True = repeats an earlier row):
0    False
1    False
2     True
3    False
dtype: bool

Age after filling with the median:
0    25.0
1    28.0
2    28.0
3    31.0
Name: age, dtype: float64


**Line by line**
- `np.nan`: NumPy's "not a number". This is how a missing value is written.
- `toy.isna()`: a table of True/False. True means missing.
- `.isna().sum()`: True counts as 1, so the sum counts the missing values per column.
- `toy.duplicated()`: marks a row True if an earlier row is exactly the same. The *first* copy is not marked.
- `fillna(value)`: replaces missing values with `value`. `median()` ignores missing values when it computes.

**How to read the output**
- `isna()` finds Ben's two missing ages but **does not flag Cara's city `" "`**. A blank space is a real text value, so it looks filled. This is the *hidden missing value* trap.
- The second Ben row is marked as a duplicate.
- The median of 25 and 31 is 28, so both missing ages become 28.

## Step 3: Find missing values in the real data

In [3]:
print("Missing values per column (isna):")
print(df.isna().sum().sum(), "missing values in total")

Missing values per column (isna):
0 missing values in total


`isna()` says **zero**. But in notebook 01 we saw that `TotalCharges` is stored as text, which suggests blanks. Let's search for hidden missing values: values that are empty after removing spaces.

In [4]:
# For every text column, count values that are empty once spaces are stripped.
text_cols = df.select_dtypes(exclude="number").columns

hidden = {}
for col in text_cols:
    n_blank = (df[col].str.strip() == "").sum()
    if n_blank > 0:
        hidden[col] = int(n_blank)

print("Hidden blanks per column:", hidden)

Hidden blanks per column: {'TotalCharges': 11}


**Line by line**
- `select_dtypes(exclude="number")`: only the text columns.
- `df[col].str.strip()`: removes spaces from the start and end of each text value. `.str` gives access to text methods.
- `== ""`: True when nothing is left, meaning the cell was blank.
- `.sum()`: counts the Trues.

**Expected output:** `{'TotalCharges': 11}`. Only `TotalCharges` has hidden blanks, 11 of them. Nothing else needs this fix.

## Step 4: Fix `TotalCharges`

We have two problems to fix: the **hidden blanks** and the **wrong data type**.

**Why are they blank?** Let's confirm our theory from notebook 01 before choosing a fix.

In [5]:
blank_mask = df["TotalCharges"].str.strip() == ""

print("Tenure of the blank rows:", df.loc[blank_mask, "tenure"].unique())
print("Did they churn?")
print(df.loc[blank_mask, "Churn"].value_counts())

Tenure of the blank rows: [0]
Did they churn?
Churn
No    11
Name: count, dtype: int64


**How to read the output**
- All blank rows have `tenure = 0`. These customers joined this month and have not received a first bill. The blank means **"nothing billed yet"**, which is a true total of **0**.
- None of the 11 churned. So the blank is not random noise. It describes one specific group (brand-new customers).

**Decision:** compare the options from the notes.

| Option | Verdict |
|---|---|
| Drop 11 rows | Possible (11 of 7,043 is 0.2%), but we throw away real customers |
| Fill with the median | Wrong in meaning: a new customer has not paid the median |
| **Fill with 0** | Correct in meaning, and it is a *rule*, so it causes no leakage |

We choose **0**.

In [6]:
# Step A: turn blanks into proper missing values (NaN) while converting text -> number.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Step B: fill with 0, because tenure == 0 means nothing has been billed yet.
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("dtype now:", df["TotalCharges"].dtype)
print("Missing now:", df["TotalCharges"].isna().sum())
print("Rows with TotalCharges == 0:", (df["TotalCharges"] == 0).sum())

dtype now: float64
Missing now: 0
Rows with TotalCharges == 0: 11


**Line by line**
- `pd.to_numeric(..., errors="coerce")`: converts text to numbers. `" "` cannot be converted, so it becomes `NaN` instead of raising an error.
- `.fillna(0)`: replaces every `NaN` with 0. Valid here only because we checked that these customers have `tenure = 0`.
- We assign back to `df["TotalCharges"]`, otherwise the change is lost.

**Expected output:** dtype `float64`, 0 missing, and 11 rows with total 0.

**Sanity check:** a customer's total should be roughly `tenure × MonthlyCharges`. Let's make sure the fixed column behaves that way.

In [7]:
check = df[["tenure", "MonthlyCharges", "TotalCharges"]].copy()
check["expected"] = check["tenure"] * check["MonthlyCharges"]
print(check.head())
print()
print("Correlation between expected and actual total:",
      round(check["expected"].corr(check["TotalCharges"]), 3))

   tenure  MonthlyCharges  TotalCharges  expected
0       1           29.85         29.85     29.85
1      34           56.95       1889.50   1936.30
2       2           53.85        108.15    107.70
3      45           42.30       1840.75   1903.50
4       2           70.70        151.65    141.40

Correlation between expected and actual total: 1.0


The correlation is close to 1, so `TotalCharges` is consistent with `tenure` and `MonthlyCharges`. (It is not exactly equal, since prices change over time and customers get discounts.)

We just noticed something useful for later: **`TotalCharges` is largely explained by `tenure × MonthlyCharges`**, so these three columns carry overlapping information. We will meet this again in EDA and feature selection.

## Step 5: Check for duplicates

In [8]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated customerIDs:", df["customerID"].duplicated().sum())
print("Duplicates ignoring customerID:", df.drop(columns="customerID").duplicated().sum())

Fully duplicated rows: 0
Duplicated customerIDs: 0
Duplicates ignoring customerID: 22


**How to read the output**
- **0 fully duplicated rows** and **0 repeated customer IDs**: no customer is recorded twice. Nothing to drop.
- Ignoring `customerID`, there are **22** rows that match another row. Should we drop them?
  **No.** Each has a different customer ID, so they are different people who happen to have identical answers. With 19 columns of mostly Yes/No answers, such coincidences are normal. Dropping them would delete real customers.

**Lesson:** a duplicate only matters if it is the *same real-world record*. Think before you drop.

## Step 6: Check for inconsistent category values

We list the unique values of each categorical column to look for typos, mixed capitalisation or odd labels.

In [9]:
cat_cols = [c for c in df.select_dtypes(exclude="number").columns if c not in ("customerID", "Churn")]

for col in cat_cols:
    print(f"{col}: {sorted(df[col].unique())}")

gender: ['Female', 'Male']
Partner: ['No', 'Yes']
Dependents: ['No', 'Yes']
PhoneService: ['No', 'Yes']
MultipleLines: ['No', 'No phone service', 'Yes']
InternetService: ['DSL', 'Fiber optic', 'No']
OnlineSecurity: ['No', 'No internet service', 'Yes']
OnlineBackup: ['No', 'No internet service', 'Yes']
DeviceProtection: ['No', 'No internet service', 'Yes']
TechSupport: ['No', 'No internet service', 'Yes']
StreamingTV: ['No', 'No internet service', 'Yes']
StreamingMovies: ['No', 'No internet service', 'Yes']
Contract: ['Month-to-month', 'One year', 'Two year']
PaperlessBilling: ['No', 'Yes']
PaymentMethod: ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']


**Line by line**
- A list comprehension builds the list of text columns, skipping the ID and the target.
- `df[col].unique()`: the distinct values in that column.
- `sorted(...)`: puts them in alphabetical order to make them easier to read.

**What we see**
- No typos, no mixed capitalisation, no stray spaces. Values are consistent.
- Something to **note** (but not change): several columns contain `"No internet service"` or `"No phone service"`. These mean the same as `"No"` for that feature: a customer without internet cannot have online security. They are not errors, but they do repeat information already in `InternetService` and `PhoneService`. We keep them for now and revisit in the feature engineering notebook.

## Step 7: Verify and save

Always re-check before saving. A good cleaning step ends with proof, not hope.

In [10]:
assert df.isna().sum().sum() == 0, "There are still missing values"
assert df["TotalCharges"].dtype == "float64", "TotalCharges is not numeric"
assert df.duplicated().sum() == 0, "There are duplicate rows"

print("All checks passed.")
df.info()

All checks passed.


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [11]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)     # index=False: do not write the row numbers as an extra column
print("Saved to", CLEAN_PATH.resolve())

Saved to C:\Users\Acer\ml all concept\data\processed\telco_clean.csv


**Line by line**
- `assert condition, "message"`: stops the notebook with an error if the condition is False. It is a cheap safety net.
- `df.to_csv(path, index=False)`: writes the DataFrame to a CSV. Without `index=False` Pandas adds an unwanted extra column of row numbers.
- `mkdir(parents=True, exist_ok=True)`: creates the folder if it is missing, with no error if it exists.

**Expected output:** `All checks passed.`, an info table where `TotalCharges` is `float64` and every column has 7043 non-null values, and the saved path.

**Note:** `data/processed/` is ignored by git. Anyone who clones the repo recreates this file by running this notebook.

## Summary of what we did

| Problem | Finding | Action |
|---|---|---|
| Missing values | 11 hidden blanks in `TotalCharges` (new customers, `tenure = 0`) | Converted to number, filled with 0 (rule-based) |
| Data types | `TotalCharges` stored as text | Converted to `float64` |
| Duplicates | None (22 look-alikes with different IDs) | Kept all rows |
| Inconsistent values | None | Nothing to change |

**Next:** `03_eda`: explore the cleaned data with statistics and charts to learn *which features relate to churn*.

---
## Exercise

1. Create a small DataFrame of your own with one real missing value, one hidden blank `" "` and one duplicate row. Find each one using the tools from Step 2.
2. In the real data, what percentage of customers have `tenure` equal to 0? Does that match the 11 rows?
3. **Think:** if 500 customers had a missing `MonthlyCharges` instead, would filling with 0 be sensible? What would you use instead, and *when* in the workflow would you compute it? (Hint: re-read section 7 of the notes.)

In [4]:
# Exercise 1
import pandas as pd 
import numpy as np 
data = {
    "Name": ["Ram", "Hari", "Sita", "Ram"],
    "Age": [22, np.nan, 25, 22],
    "City": ["Kathmandu", " ", "Pokhara", "Kathmandu"]
}
df = pd.DataFrame(data)

print(df)

df.duplicated()

df = df.replace(r"^\s*$", np.nan, regex=True)

df.isnull().sum()

   Name   Age       City
0   Ram  22.0  Kathmandu
1  Hari   NaN           
2  Sita  25.0    Pokhara
3   Ram  22.0  Kathmandu


Name    0
Age     1
City    1
dtype: int64